In [ ]:
import time, math
import numpy as np
import PIL.Image
from pynq import Overlay, MMIO, allocate
import svo_builder

BITSTREAM = '/home/xilinx/jupyter_notebooks/svo_system.bit'
IMG_W, IMG_H = 320, 240
BYTES_PER_PIXEL = 4   # 32-bit XRGB

def to_q16(f): return int(f * 65536) & 0xFFFF_FFFF

# Pack RGB so that ip.write() stores R in bits [23:16], G in [15:8], B in [7:0]
# (matches traversal tdata layout: tdata[23:16]=R, tdata[15:8]=G, tdata[7:0]=B)
def pack_rgb(r, g, b): return ((int(r)&0xFF)<<16) | ((int(g)&0xFF)<<8) | (int(b)&0xFF)

def normalise(v):
    l = math.sqrt(sum(x**2 for x in v))
    return [x/l for x in v] if l > 1e-9 else v
def cross(a, b):
    return [a[1]*b[2]-a[2]*b[1], a[2]*b[0]-a[0]*b[2], a[0]*b[1]-a[1]*b[0]]

# ---------------------------------------------------------------------------
# Load bitstream
# ---------------------------------------------------------------------------
print('Loading bitstream ...')
ol = Overlay(BITSTREAM)
ip = ol.top_0

# ---------------------------------------------------------------------------
# Configure VDMA S2MM via direct MMIO (AXI VDMA v6.3, PG020)
# S2MM register offsets:
#   0x30  S2MM_VDMACR       control (bit0=RS, bit1=Circular_Park, bit2=Reset)
#   0x34  S2MM_VDMASR       status  (bit0=Halted)
#   0xA0  S2MM_VSIZE        lines   (write LAST — arms channel)
#   0xA4  S2MM_HSIZE        bytes per line
#   0xA8  S2MM_FRMDLY_STRIDE
#   0xAC/0xB0/0xB4  S2MM_SA1/2/3  (C_NUM_FSTORES=3)
# ---------------------------------------------------------------------------
VDMA_BASE = ol.ip_dict['axi_vdma_0']['phys_addr']
vdma = MMIO(VDMA_BASE, 0x1000)

HSIZE  = IMG_W * BYTES_PER_PIXEL
STRIDE = IMG_W * BYTES_PER_PIXEL

frame_buf  = allocate(shape=(IMG_H, IMG_W, BYTES_PER_PIXEL), dtype=np.uint8)
frame_phys = frame_buf.physical_address

# Reset S2MM and wait for self-clear
vdma.write(0x30, 0x4)
while vdma.read(0x30) & 0x4: pass

# All 3 frame buffer start addresses
vdma.write(0xAC, frame_phys)
vdma.write(0xB0, frame_phys)
vdma.write(0xB4, frame_phys)

vdma.write(0xA8, STRIDE)
vdma.write(0xA4, HSIZE)
vdma.write(0x30, 0x3)       # RS=1, Circular_Park=1
vdma.write(0xA0, IMG_H)     # VSIZE last — arms channel, tready asserts

s2mm_sr = vdma.read(0x34)
print(f'VDMA S2MM ready: phys=0x{frame_phys:08X}, status=0x{s2mm_sr:08X}')
if s2mm_sr & 0x1:
    print('WARNING: S2MM is Halted — check HSIZE/VSIZE/addresses and re-run')

In [ ]:
# ---------------------------------------------------------------------------
# Build and upload SVO
# ---------------------------------------------------------------------------
print('Building SVO ...')
grid  = svo_builder.build_world()
root  = svo_builder.build_svo(grid)
nodes = svo_builder.flatten_svo(root)
words = svo_builder.serialise_nodes(nodes)
print(f'  {len(nodes)} nodes -> {len(words)} words ({len(words)*4} bytes)')

print('Uploading SVO to BRAM ...')
ip.write(0x48, 0)          # reset write address to 0
for w in words:
    ip.write(0x4C, w)      # each write auto-increments the address
print('  Upload done.')

In [ ]:
# ---------------------------------------------------------------------------
# Sky colour  (register 0x68: bits [23:16]=R, [15:8]=G, [7:0]=B)
# Hit colour in SHADE_MODE=0 is always white (0xFFFFFF).
# ---------------------------------------------------------------------------
SKY_R, SKY_G, SKY_B = 135, 206, 235   # light blue
ip.write(0x68, pack_rgb(SKY_R, SKY_G, SKY_B))

# ---------------------------------------------------------------------------
# Camera setup
# pos = (32, 40, -20), looking toward world centre (32, 4, 32)
# right-handed coord: +Y up, camera faces +Z into the scene
# ---------------------------------------------------------------------------
pos   = [32.0, 40.0, -20.0]
fwd   = normalise([32.0 - pos[0], 4.0 - pos[1], 32.0 - pos[2]])
right = normalise(cross(fwd, [0, 1, 0]))
up    = cross(right, fwd)

# fov_scale = tan(fov/2) / half_width  (per-pixel ray spread)
fov_scale = math.tan(math.radians(60) / 2) / (IMG_W / 2)

# Camera registers (Q16.16 fixed-point)
ip.write(0x08, to_q16(pos[0]))
ip.write(0x0C, to_q16(pos[1]))
ip.write(0x10, to_q16(pos[2]))

ip.write(0x14, to_q16(right[0]))
ip.write(0x18, to_q16(right[1]))
ip.write(0x1C, to_q16(right[2]))

ip.write(0x20, to_q16(up[0]))
ip.write(0x24, to_q16(up[1]))
ip.write(0x28, to_q16(up[2]))

ip.write(0x2C, to_q16(fwd[0]))
ip.write(0x30, to_q16(fwd[1]))
ip.write(0x34, to_q16(fwd[2]))

ip.write(0x38, to_q16(fov_scale))

print('Camera:')
print(f'  pos   = {[round(x,3) for x in pos]}')
print(f'  fwd   = {[round(x,3) for x in fwd]}')
print(f'  right = {[round(x,3) for x in right]}')
print(f'  up    = {[round(x,3) for x in up]}')
print(f'  scale = {fov_scale:.6f}')

In [ ]:
# ---------------------------------------------------------------------------
# Re-arm VDMA S2MM before each render
# ---------------------------------------------------------------------------
vdma.write(0x30, 0x4)               # reset S2MM
while vdma.read(0x30) & 0x4: pass   # wait for self-clear
vdma.write(0xAC, frame_phys)
vdma.write(0xB0, frame_phys)
vdma.write(0xB4, frame_phys)
vdma.write(0xA8, STRIDE)
vdma.write(0xA4, HSIZE)
vdma.write(0x30, 0x3)               # RS=1, Circular_Park=1
vdma.write(0xA0, IMG_H)             # VSIZE last — arms channel, asserts tready
s2mm_sr = vdma.read(0x34)
if s2mm_sr & 0x1:
    print(f'WARNING: S2MM still halted after re-arm (status=0x{s2mm_sr:08X})')

# FSM state names (sequential encoding forced by (* fsm_encoding="sequential" *))
FSM_STATES = {
    0:'S_IDLE', 1:'S_RAY_SETUP', 2:'S_ROOT_SLAB', 3:'S_ENTER_NODE',
    4:'S_BRAM_WAIT', 5:'S_CHECK_CHILD', 6:'S_EMPTY', 7:'S_SOLID',
    8:'S_MIXED', 9:'S_POP_STACK', 10:'S_MISS', 11:'S_WAIT_SHADE',
    12:'S_WRITE_PIXEL', 13:'S_NEXT_PIXEL'
}

# ---------------------------------------------------------------------------
# Trigger render and poll with live debug
# 0x78 register: bits[3:0]=FSM state, bit[4]=tvalid, bit[5]=tready
#   state=12 + tvalid=1 + tready=0  →  stuck waiting for VDMA (backpressure)
#   state=11 + tvalid=0             →  stuck in S_WAIT_SHADE (shade pipeline hang)
# ---------------------------------------------------------------------------
print('Triggering render ...')
t0 = time.time()
ip.write(0x00, 1)

# Wait for busy=1 (render started)
while not (ip.read(0x04) & 0x1):
    if time.time() - t0 > 0.5:
        print('ERROR: busy never went high — trigger did not fire'); break

# Poll busy with periodic debug prints
last_print = time.time()
while ip.read(0x04) & 0x1:
    now = time.time()
    if now - last_print >= 1.0:
        dbg    = ip.read(0x78)
        pxpy   = ip.read(0x7C)
        state  = dbg & 0xF
        tvalid = (dbg >> 4) & 1
        tready = (dbg >> 5) & 1
        px     = (pxpy >> 8) & 0x1FF
        py     = pxpy & 0xFF
        print(f'  t={now-t0:.1f}s  state={state} ({FSM_STATES.get(state,"?")})'
              f'  px={px}  py={py}  tvalid={tvalid}  tready={tready}')
        if now - t0 > 10.0:
            print('TIMEOUT after 10 s — aborting poll')
            break
        last_print = now
    time.sleep(0.001)

elapsed = time.time() - t0
print(f'Done in {elapsed:.3f} s  ({1/elapsed:.2f} FPS)')

time.sleep(0.05)
s2mm_sr = vdma.read(0x34)
if s2mm_sr & 0x70:
    print(f'WARNING: VDMA error bits set: status=0x{s2mm_sr:08X}')

# Invalidate cache and read frame
frame_buf.invalidate()
frame_rgb = np.array(frame_buf[:, :, [2, 1, 0]])
image = PIL.Image.fromarray(frame_rgb, 'RGB')

print(f'Non-zero pixels: {np.count_nonzero(frame_buf[:,:,:3])}')
print(f'White (hit) pixels: {np.sum(np.all(frame_rgb == 255, axis=2))}')

In [ ]:
# Display inline
image

In [ ]:
# Optionally save the image
image.save('/home/xilinx/jupyter_notebooks/render.png')
print('Saved to render.png')

In [ ]:
# Stop VDMA and free frame buffer
vdma.write(0x30, 0x0)   # RS=0 (stop)
frame_buf.freebuffer()